In [3]:
from pymongo import MongoClient

# Verbindung zur MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client['Espana']
collection = db['ciudad']

pipeline = [
    # 1. Sicherstellen, dass die Felder existieren
    {
        "$match": {
            "Com autónoma": "Madrid",
            "Temperature high": { "$exists": True },
            "Temperature low": { "$exists": True },
            "rainfall mm": {"$exists": True}
        }
    },

    # 2. Felder in Arrays transformieren
    {
        "$project": {
            "Ciudad": 1,
            "Provincia": 1,
            "Población": 1,
            "Densidad": 1,
            "Altitud": 1,
            "high_arr": { "$objectToArray": "$Temperature high" },
            "low_arr": { "$objectToArray": "$Temperature low" },
            "rain_arr": { "$objectToArray": { "$ifNull": ["$rainfall mm", {}] } }
            
        }
    },

    # 3. Berechnungen (Durchschnitte, Min, Max)
    {
        "$addFields": {
            "avg_high": { "$avg": "$high_arr.v" },
            "avg_low": { "$avg": "$low_arr.v" },
            "max_temp": { "$max": "$high_arr.v" },
            "min_temp": { "$min": "$low_arr.v" },
            "avg_rain": { "$avg": "$rain_arr.v" },
            "water_arr": { "$objectToArray": { "$ifNull": ["$water", {}] } }
        }
    },

    # 4. Spanne und Wasser-Durchschnitt
    {
        "$addFields": {
            "avg_water": { "$avg": "$water_arr.v" },
            "amplitude": { "$subtract": ["$max_temp", "$min_temp"] }
        }
    },
    { "$sort": { "avg_high": -1 } }
]

results = collection.aggregate(pipeline)

header = f"{'Ciudad':<30} | {'Provincia':<25}  | {'Población':<10} | {'Densidad': <11} | {'Altitud':<5} |  {'High':<6} | {'Low':<6} | {'Spanne':<7} | {'Regen':<7} "
print(header)
print("-" * len(header))

for res in results:
    ciudad = res.get('Ciudad', 'Unbekannt')
    Provincia = res.get('Provincia', 'Unbekannt')
    Población = res.get('Población', 'Unbekannt')
    Densidad =  res.get('Densidad', 'Unbekannt')
    Altitud =  res.get('Altitud', 'Unbekannt')
    
    # Sicherer Fallback: Wenn ein Wert None ist, wird 0 verwendet
    a_high = round(res.get('avg_high') or 0, 1)
    a_low  = round(res.get('avg_low') or 0, 1)
    amp    = round(res.get('amplitude') or 0, 1)
    rain   = round(res.get('avg_rain') or 0, 1)
    
    w_val  = res.get('avg_water')
    water_str = f"{round(w_val, 1):>6}" if w_val is not None else "  ---  "

    print(f"{ciudad:<30} |  {Provincia:<25} |  {Población:>10} | {Densidad:>11} | {Altitud:>5} | {a_high:>6} | {a_low:>6} | {amp:>7} | {rain:>7}")

Ciudad                         | Provincia                  | Población  | Densidad    | Altitud |  High   | Low    | Spanne  | Regen   
----------------------------------------------------------------------------------------------------------------------------------------
Villa del Prado                |  Madrid                    |        7743 |       98.74 |   510 |   21.1 |    8.8 |      32 |    23.4
San Martín de Valdeiglesias    |  Madrid                    |        9324 |       80.73 |   681 |   19.8 |    7.8 |      32 |    23.6
Ambite                         |  Madrid                    |         732 |       28.15 |   682 |   19.2 |    6.6 |      32 |    22.9
